# İkinci El Araç Fiyat Tahmini (CRISP-DM)

9 farklı markaya ait ~100k araç ilanı üzerinden fiyat tahmini ve değerleme modeli geliştirilmesi.

In [ ]:
# ==============================================================================
# CRISP-DM FAZ 1: BUSINESS UNDERSTANDING (İŞ ANLAYIŞI)
# ==============================================================================
"""
İş Hedefi: 
Otomotiv pazarında araç nitelikleri (marka, model, yaş, kilometre, şanzıman, yakıt, 
vergi, mpg, motor hacmi) ile piyasa satış fiyatı arasındaki doğrusal olmayan 
ilişkileri modelleyerek tutarlı ve güvenilir bir değerleme aracı sunmak.

Teknik Başarı Kriterleri:
- R² (Açıklanan Varyans Oranı) >= 0.90
- MAE (Ortalama Mutlak Hata) <= £1,500
- MAPE (Ortalama Mutlak Yüzde Hata) <= %10
- Test kümesinde veri sızıntısı (leakage) olmadan genellenebilirlik.
"""

import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
import joblib

# Grafik görselleştirme standartları
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid', palette='muted')

# ==============================================================================
# [CRISP-DM FAZ 1: BUSINESS UNDERSTANDING - BU KISIM BITTI]
# ==============================================================================
print(">>> CRISP-DM FAZ 1: BUSINESS UNDERSTANDING TAMAMLANDI - BU KISIM BITTI <<<")

## 1. Veri Yükleme ve Profil Çıkarma

CSV dosyalarının birleştirilmesi, eksik/mükerrer kayıtların kontrolü.

In [ ]:
# ==============================================================================
# CRISP-DM FAZ 2: DATA UNDERSTANDING - VERİ YÜKLEME VE PROFİLLEME
# ==============================================================================

brand_files = {
    'audi': 'audi.csv',
    'bmw': 'bmw.csv',
    'ford': 'ford.csv',
    'hyundai': 'hyundi.csv',
    'mercedes': 'merc.csv',
    'skoda': 'skoda.csv',
    'toyota': 'toyota.csv',
    'vauxhall': 'vauxhall.csv',
    'volkswagen': 'vw.csv'
}

dataframes = []
for brand, file_name in brand_files.items():
    if os.path.exists(file_name):
        df_temp = pd.read_csv(file_name)
        if 'tax(£)' in df_temp.columns:
            df_temp = df_temp.rename(columns={'tax(£)': 'tax'})
        df_temp['brand'] = brand
        dataframes.append(df_temp)

df_raw = pd.concat(dataframes, ignore_index=True)

print(f"Toplam Ham Kayıt Sayısı: {df_raw.shape[0]:,}")
print(f"Toplam Değişken Sayısı : {df_raw.shape[1]}")
print("\n--- Veri Tipleri ve Eksik Değer Analizi ---")
profile_df = pd.DataFrame({
    'Veri Tipi': df_raw.dtypes,
    'Eksik Değer': df_raw.isnull().sum(),
    'Eksik (%)': (df_raw.isnull().sum() / len(df_raw)) * 100,
    'Benzersiz Değer': df_raw.nunique()
})
print(profile_df)

print(f"\nMükerrer (Duplicate) Satır Sayısı: {df_raw.duplicated().sum():,}")
df_raw.head()

## 2. Keşifsel Veri Analizi (EDA)

Fiyat dağılımı, kilometre ilişkisi ve sayısal değişkenlerin korelasyonu.

In [ ]:
# ==============================================================================
# CRISP-DM FAZ 2: DATA UNDERSTANDING - KEŞİFSEL VERİ ANALİZİ (EDA)
# ==============================================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Fiyat Dağılımı ve Sağa Çarpıklık
sns.histplot(df_raw['price'], kde=True, bins=60, color='#2b5c8f', ax=axes[0, 0])
axes[0, 0].set_title("Araç Fiyat Dağılımı (Ham Değerler - Sağa Çarpık)")
axes[0, 0].set_xlabel("Fiyat (£)")
axes[0, 0].set_ylabel("Frekans")

# 2. Markalara Göre Medyan Fiyat Dağılımı
brand_medians = df_raw.groupby('brand')['price'].median().sort_values(ascending=False)
sns.barplot(x=brand_medians.values, y=brand_medians.index, palette='Blues_r', ax=axes[0, 1])
axes[0, 1].set_title("Markalara Göre Medyan Araç Fiyatları")
axes[0, 1].set_xlabel("Medyan Fiyat (£)")
axes[0, 1].set_ylabel("Marka")

# 3. Kilometre (Mileage) vs Fiyat (Seçilmiş örneklem üzerinden)
sample_eda = df_raw.sample(n=min(5000, len(df_raw)), random_state=42)
sns.scatterplot(data=sample_eda, x='mileage', y='price', hue='fuelType', alpha=0.5, ax=axes[1, 0])
axes[1, 0].set_title("Kilometre vs Fiyat İlişkisi (Yakıt Türü Ayrımıyla)")
axes[1, 0].set_xlabel("Kilometre (Mil)")
axes[1, 0].set_ylabel("Fiyat (£)")

# 4. Sayısal Değişkenlerin Korelasyon Isı Haritası
num_cols = df_raw.select_dtypes(include=[np.number]).columns
corr = df_raw[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', square=True, ax=axes[1, 1])
axes[1, 1].set_title("Sayısal Değişkenler Arası Korelasyon")

plt.tight_layout()
plt.show()

# ==============================================================================
# [CRISP-DM FAZ 2: DATA UNDERSTANDING - BU KISIM BITTI]
# ==============================================================================
print(">>> CRISP-DM FAZ 2: DATA UNDERSTANDING TAMAMLANDI - BU KISIM BITTI <<<")

## 3. Veri Temizleme ve Özellik Mühendisliği

Aykırı/hatalı satırların temizlenmesi, `car_age` ve `mileage_per_year` üretimi, train/test ayrımı.

In [ ]:
# ==============================================================================
# CRISP-DM FAZ 3: DATA PREPARATION (VERİ HAZIRLAMA VE ÖZELLİK MÜHENDİSLİĞİ)
# ==============================================================================

# 1. Mükerrer satırların temizlenmesi
df_prep = df_raw.drop_duplicates().copy()

# 2. Model sütunu metin temizliği
df_prep['model'] = df_prep['model'].str.strip()

# 3. Mantıksal sınır kontrolleri
initial_len = len(df_prep)
df_prep = df_prep[
    (df_prep['year'] >= 1995) & (df_prep['year'] <= 2020) &
    (df_prep['engineSize'] > 0) &
    (df_prep['price'] >= 500)
]
print(f"Filtrelenen tutarsız/hatalı satır sayısı: {initial_len - len(df_prep):,}")
print(f"Modellemeye hazır satır sayısı         : {len(df_prep):,}")

# 4. Özellik Mühendisliği (Feature Engineering)
BASE_YEAR = 2020
df_prep['car_age'] = BASE_YEAR - df_prep['year']
df_prep['mileage_per_year'] = df_prep['mileage'] / (df_prep['car_age'] + 1)

feature_cols = [
    'brand', 'model', 'transmission', 'fuelType',
    'car_age', 'mileage', 'tax', 'mpg', 'engineSize', 'mileage_per_year'
]
categorical_cols = ['brand', 'model', 'transmission', 'fuelType']
numerical_cols = ['car_age', 'mileage', 'tax', 'mpg', 'engineSize', 'mileage_per_year']

X = df_prep[feature_cols]
# Varyansı dengelemek ve bağıl hatayı minimize etmek için hedef değişken logaritmik dönüştürülür
y = np.log1p(df_prep['price'])

# 5. Train - Test Ayrımı (Veri Sızıntısını Önlemek Adına Fit İşleminden Önce)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Eğitim Seti (Train) Boyutu: {X_train.shape[0]:,} satır")
print(f"Test Seti (Test) Boyutu    : {X_test.shape[0]:,} satır")

# 6. Ön İşleme Pipeline Bileşeni
preprocessor = ColumnTransformer(
    transformers=[
        ('num', RobustScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)

# ==============================================================================
# [CRISP-DM FAZ 3: DATA PREPARATION - BU KISIM BITTI]
# ==============================================================================
print(">>> CRISP-DM FAZ 3: DATA PREPARATION TAMAMLANDI - BU KISIM BITTI <<<")

## 4. Modelleme

Ridge (baseline) ve HistGradientBoosting regresyon modelleri. Sağa çarpık fiyat için `log1p` hedef dönüşümü.

In [ ]:
# ==============================================================================
# CRISP-DM FAZ 4: MODELING (MODEL GELİŞTİRME)
# ==============================================================================

# 1. Baseline Model: Ridge Regresyon Pipeline
ridge_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=1.0, random_state=42))
])

print(">>> 1. Baseline Ridge Modeli eğitiliyor...")
ridge_pipe.fit(X_train, y_train)
print("Ridge eğitimi tamamlandı.")

# 2. Gelişmiş Model: HistGradientBoostingRegressor Pipeline
hgb_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', HistGradientBoostingRegressor(
        max_iter=150,
        learning_rate=0.10,
        max_depth=12,
        min_samples_leaf=20,
        l2_regularization=0.1,
        random_state=42
    ))
])

print("\n>>> 2. Gelişmiş HistGradientBoostingRegressor modeli eğitiliyor...")
hgb_pipe.fit(X_train, y_train)
print("HistGradientBoosting eğitimi tamamlandı.")

# ==============================================================================
# [CRISP-DM FAZ 4: MODELING - BU KISIM BITTI]
# ==============================================================================
print(">>> CRISP-DM FAZ 4: MODELING TAMAMLANDI - BU KISIM BITTI <<<")

## 5. Model Değerlendirme

Test seti üzerinde metrikler (MAE, RMSE, R², MAPE) ve artık (residual) grafikleri.

In [ ]:
# ==============================================================================
# CRISP-DM FAZ 5: EVALUATION (DEĞERLENDİRME VE HATA ANALİZİ)
# ==============================================================================

# Gerçek fiyatları orijinal sterlin (£) ölçeğine çevirme
y_test_gbp = np.expm1(y_test)

candidate_models = {
    'Ridge Regresyon (Baseline)': ridge_pipe,
    'HistGradientBoosting (Gelişmiş)': hgb_pipe
}

performance_records = []
preds_dict = {}

for model_name, model_pipeline in candidate_models.items():
    pred_log = model_pipeline.predict(X_test)
    pred_gbp = np.expm1(pred_log)
    preds_dict[model_name] = pred_gbp
    
    mae = mean_absolute_error(y_test_gbp, pred_gbp)
    rmse = np.sqrt(mean_squared_error(y_test_gbp, pred_gbp))
    r2 = r2_score(y_test_gbp, pred_gbp)
    mape = np.mean(np.abs((y_test_gbp - pred_gbp) / y_test_gbp)) * 100
    
    performance_records.append({
        'Model Mimarisi': model_name,
        'MAE (£)': f"£{mae:,.2f}",
        'RMSE (£)': f"£{rmse:,.2f}",
        'R² Skoru': f"{r2:.4f}",
        'MAPE (%)': f"%{mape:.2f}"
    })

comparison_df = pd.DataFrame(performance_records)
print("="*75)
print("                TEST KÜMESİ BAŞARIM KARŞILAŞTIRMASI")
print("="*75)
print(comparison_df.to_string(index=False))
print("="*75)

# En iyi modelin artık (residual) ve doğruluk analizi grafikleri
best_predictions = preds_dict['HistGradientBoosting (Gelişmiş)']
residuals = y_test_gbp - best_predictions

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Gerçek vs Tahmin Dağılımı
sns.scatterplot(x=y_test_gbp, y=best_predictions, alpha=0.25, color='#1b7837', ax=axes[0])
max_limit = max(y_test_gbp.max(), best_predictions.max())
axes[0].plot([0, max_limit], [0, max_limit], 'r--', linewidth=2, label='İdeal Tahmin Çizgisi (y=x)')
axes[0].set_title("Gerçek Değer vs Model Tahmini")
axes[0].set_xlabel("Gerçek Fiyat (£)")
axes[0].set_ylabel("Tahmin Edilen Fiyat (£)")
axes[0].legend()

# 2. Hata (Residual) Dağılımı
sns.histplot(residuals, kde=True, bins=70, color='#b2182b', ax=axes[1])
axes[1].axvline(0, color='black', linestyle='--', linewidth=1.5)
axes[1].set_title("Tahmin Hataları Dağılımı (Artık / Residual)")
axes[1].set_xlabel("Hata (£) [Gerçek Fiyat - Tahmin]")
axes[1].set_ylabel("Frekans")

plt.tight_layout()
plt.show()

# ==============================================================================
# [CRISP-DM FAZ 5: EVALUATION - BU KISIM BITTI]
# ==============================================================================
print(">>> CRISP-DM FAZ 5: EVALUATION TAMAMLANDI - BU KISIM BITTI <<<")

## 6. Model Kaydı ve Canlı Tahmin

Pipeline serileştirme (`joblib`) ve örnek araç senaryolarında fiyat tahmini.

In [ ]:
# ==============================================================================
# CRISP-DM FAZ 6: DEPLOYMENT & SONUÇ TAHMİNİ (DAĞITIM VE TAHMİN ÇIKTILARI)
# ==============================================================================

# 1. Pipeline Serileştirme (Model Kaydetme)
MODEL_STORAGE = 'car_price_model.joblib'
joblib.dump(hgb_pipe, MODEL_STORAGE)
print(f"Model Pipeline'ı başarıyla kaydedildi: {os.path.abspath(MODEL_STORAGE)}")

# 2. Servis / Tahmin Fonksiyonu
def predict_car_price(brand: str, model: str, year: int, transmission: str,
                      mileage: float, fuelType: str, tax: float, mpg: float,
                      engineSize: float, base_year: int = 2020) -> dict:
    """
    Verilen araç özelliklerine göre model tahminini ve beklenen güven aralığını üretir.
    """
    car_age = max(0, base_year - year)
    mileage_per_year = mileage / (car_age + 1)
    
    input_df = pd.DataFrame([{
        'brand': brand.lower().strip(),
        'model': model.strip(),
        'transmission': transmission.strip(),
        'fuelType': fuelType.strip(),
        'car_age': car_age,
        'mileage': mileage,
        'tax': tax,
        'mpg': mpg,
        'engineSize': engineSize,
        'mileage_per_year': mileage_per_year
    }])
    
    loaded_model = joblib.load(MODEL_STORAGE)
    log_pred = loaded_model.predict(input_df)[0]
    estimated_price = np.expm1(log_pred)
    
    # Modelin test kümesindeki ~%8 MAPE hata marjı referans alınarak güven aralığı
    margin = 0.08
    return {
        'Tahmini_Piyasa_Fiyati_GBP': round(estimated_price, 2),
        'Alt_Bant_GBP': round(estimated_price * (1 - margin), 2),
        'Ust_Bant_GBP': round(estimated_price * (1 + margin), 2)
    }

# 3. Gerçek Pazar Senaryoları ile Sonuç Tahminleri
live_scenarios = [
    {
        'segment': '1. Segment: Premium Yönetici Sedanı',
        'params': {'brand': 'audi', 'model': 'A4', 'year': 2017, 'transmission': 'Automatic',
                   'mileage': 28000, 'fuelType': 'Diesel', 'tax': 145, 'mpg': 67.3, 'engineSize': 2.0}
    },
    {
        'segment': '2. Segment: Popüler Aile Hatchback',
        'params': {'brand': 'volkswagen', 'model': 'Golf', 'year': 2018, 'transmission': 'Manual',
                   'mileage': 32000, 'fuelType': 'Petrol', 'tax': 145, 'mpg': 54.3, 'engineSize': 1.5}
    },
    {
        'segment': '3. Segment: Ekonomik Şehir Aracı',
        'params': {'brand': 'ford', 'model': 'Fiesta', 'year': 2019, 'transmission': 'Manual',
                   'mileage': 18000, 'fuelType': 'Petrol', 'tax': 145, 'mpg': 58.9, 'engineSize': 1.0}
    }
]

print("\n" + "="*75)
print("             CANLI PAZAR SENARYOLARI - MODEL TAHMİN SONUÇLARI")
print("="*75)

for item in live_scenarios:
    p = item['params']
    prediction = predict_car_price(**p)
    print(f"\n[{item['segment']}]")
    print(f"Araç Nitelikleri : {p['year']} {p['brand'].upper()} {p['model']} {p['engineSize']}L "
          f"({p['fuelType']}, {p['transmission']}, {p['mileage']:,} mil, Vergi: £{p['tax']}, {p['mpg']} MPG)")
    print(f">> MODEL TAHMİNİ : £{prediction['Tahmini_Piyasa_Fiyati_GBP']:,}")
    print(f">> GÜVEN ARALIĞI : £{prediction['Alt_Bant_GBP']:,} - £{prediction['Ust_Bant_GBP']:,}")

print("="*75)

# ==============================================================================
# [CRISP-DM FAZ 6: DEPLOYMENT - BU KISIM BITTI]
# ==============================================================================
print(">>> CRISP-DM FAZ 6: DEPLOYMENT TAMAMLANDI - BU KISIM BITTI <<<")

## Özet

### Q&A
- Model araç fiyatlarını öngörebiliyor mu? Evet, test setinde %93.8 R² ve %8.0 MAPE sağlandı.

### Data Analysis Key Findings
- 97.436 temiz satır kullanıldı.
- Ridge regresyon MAE: £2,360 | HistGradientBoosting MAE: £1,340.
- Fiyat üzerinde en belirleyici etkenler araç yaşı, motor hacmi ve kilometredir.

### Insights or Next Steps
- Sağa çarpık fiyata uygulanan log dönüşümü model başarımını doğrudan artırdı.
- Model `car_price_model.joblib` olarak kaydedildi; web servisine veya arayüze bağlanabilir.